Τι περιλαμβανει το νεο section

Μικρο stratified subset (default: 150/train ανα κλαση = 1500 train, 50/test ανα κλαση = 500 test) για να ειναι εφικτο το KPCA.

HOG extraction στο subset.

Small grid search για KPCA:

kpca__n_components = [32, 64, 128]

kpca__gamma = [0.5*base, base, 2*base, 5*base], οπου base = 1 / n_features(HOG)

Επιλεγει τα καλυτερα KPCA params με pipeline (Scaler → KPCA → LDA → NCC) (για να ειναι γρηγορο).

Με τα best KPCA params “κλειδωνει” reducer Scaler → KPCA(best) → LDA και μετα κανει:

GridSearchCV για kNN

GridSearchCV για NCC
στα μειωμενα χαρακτηριστικα (KPCA+LDA).

## HOG + KPCA + LDA (small subset) → kNN / NCC

Σε αυτο το section κανουμε:
- HOG feature extraction
- small grid search για KernelPCA (RBF) παραμετρους (n_components, gamma)
- LDA μετα απο KPCA (max 9 dims για CIFAR-10)
- ταξινομηση με kNN και Nearest Class Centroid στα μειωμενα χαρακτηριστικα

Σημειωση: το KPCA εχει κοστος ~O(N^2) σε μνημη/χρονο (kernel matrix), αρα δουλευουμε σε μικρο stratified subset.


Load libraries

In [1]:
from data_loader import load_cifar10
import matplotlib.pyplot as plt
import numpy as np
import time

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
from skimage.color import rgb2gray
from sklearn.decomposition import KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline

In [2]:
def select_stratified_subset(
    X_train_flat, y_train,
    X_test_flat, y_test,
    n_train_per_class, n_test_per_class,
    random_state=0):
    
    num_classes = len(label_names)

    rng = np.random.default_rng(random_state)
    train_indices = []

    for c in range(num_classes):
        class_idxs = np.where(y_train == c)[0]
        if len(class_idxs) < n_train_per_class:
            raise ValueError(
                f"Not enough train samples for class {c}: "
                f"have {len(class_idxs)}, requested {n_train_per_class}"
            )
        chosen = rng.choice(class_idxs, size=n_train_per_class, replace=False)
        train_indices.append(chosen)

    train_indices = np.concatenate(train_indices)
    rng.shuffle(train_indices)  

    X_train_sub = X_train_flat[train_indices]
    y_train_sub = y_train[train_indices]

    test_indices = []

    for c in range(num_classes):
        class_idxs = np.where(y_test == c)[0]
        if len(class_idxs) < n_test_per_class:
            raise ValueError(
                f"Not enough test samples for class {c}: "
                f"have {len(class_idxs)}, requested {n_test_per_class}"
            )
        chosen = rng.choice(class_idxs, size=n_test_per_class, replace=False)
        test_indices.append(chosen)

    test_indices = np.concatenate(test_indices)
    rng.shuffle(test_indices)

    X_test_sub = X_test_flat[test_indices]
    y_test_sub = y_test[test_indices]

    print("Train shape:", X_train_sub.shape, "Test shape:", X_test_sub.shape)
    return X_train_sub, y_train_sub, X_test_sub, y_test_sub

In [3]:
data = load_cifar10("cifar-10-batches-py")
X_train, y_train = data["x_train"], data["y_train"]
X_test, y_test   = data["x_test"], data["y_test"]
label_names = data["label_names"]

In [4]:
def compute_hog_batch(X):
    hog_list = []
    for img in X:
        # grayscale
        gray = rgb2gray(img)
        feats = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            visualize=False
        )
        hog_list.append(feats)
    return np.array(hog_list, dtype=np.float32)

In [5]:
results = []
# 1) Stratified subset (μικρο) για KPCA+LDA
X_train_img_kpca, y_train_img_kpca, X_test_img_kpca, y_test_img_kpca = select_stratified_subset(
    X_train, y_train,
    X_test,  y_test,
    n_train_per_class=150,   # 150*10 = 1500 train
    n_test_per_class=50,     # 50*10  = 500  test
    random_state=0
)
print("KPCA subset shapes:", X_train_img_kpca.shape, X_test_img_kpca.shape)

Train shape: (1500, 32, 32, 3) Test shape: (500, 32, 32, 3)
KPCA subset shapes: (1500, 32, 32, 3) (500, 32, 32, 3)


In [6]:
# 2) HOG extraction (KPCA subset)
print("\nExtracting HOG features for KPCA subset...")
start = time.perf_counter()
X_train_hog_kpca = compute_hog_batch(X_train_img_kpca)
X_test_hog_kpca  = compute_hog_batch(X_test_img_kpca)
hog_kpca_time = time.perf_counter() - start

print(f"HOG KPCA-subset extraction done in {hog_kpca_time:.2f} sec.")
print("HOG shapes:", X_train_hog_kpca.shape, X_test_hog_kpca.shape)

# baseline gamma around 1 / n_features
gamma_base = 1.0 / X_train_hog_kpca.shape[1]
gamma_grid = [0.5*gamma_base, gamma_base, 2*gamma_base, 5*gamma_base]
ncomp_grid = [32, 64, 128]  # small grid

print("gamma_base:", gamma_base)
print("gamma_grid:", gamma_grid)
print("n_components grid:", ncomp_grid)


Extracting HOG features for KPCA subset...
HOG KPCA-subset extraction done in 0.65 sec.
HOG shapes: (1500, 324) (500, 324)
gamma_base: 0.0030864197530864196
gamma_grid: [0.0015432098765432098, 0.0030864197530864196, 0.006172839506172839, 0.015432098765432098]
n_components grid: [32, 64, 128]


In [7]:
# 3) Small grid search για KPCA params (χρησιμοποιουμε NCC ως γρηγορο classifier)
kpca_lda_ncc_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kpca", KernelPCA(
        kernel="rbf",
        eigen_solver="randomized",
        random_state=0
    )),
    ("lda", LinearDiscriminantAnalysis(n_components=9)),
    ("clf", NearestCentroid())
])

kpca_param_grid = {
    "kpca__n_components": ncomp_grid,
    "kpca__gamma": gamma_grid,
}

print("\nRunning KPCA grid search (with LDA + NCC)...")
start = time.perf_counter()
kpca_grid = GridSearchCV(
    estimator=kpca_lda_ncc_pipe,
    param_grid=kpca_param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=1
)
kpca_grid.fit(X_train_hog_kpca, y_train_img_kpca)
kpca_grid_time = time.perf_counter() - start

print("Best KPCA params:", kpca_grid.best_params_)
print(f"Best CV accuracy: {kpca_grid.best_score_:.4f}")
print(f"KPCA grid search time: {kpca_grid_time:.2f} sec")

best_kpca_params = kpca_grid.best_params_


Running KPCA grid search (with LDA + NCC)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


Best KPCA params: {'kpca__gamma': 0.006172839506172839, 'kpca__n_components': 128}
Best CV accuracy: 0.4500
KPCA grid search time: 9.60 sec


In [8]:
# 4) Fit best reducer: StandardScaler → KPCA(best) → LDA
best_reducer = Pipeline([
    ("scaler", StandardScaler()),
    ("kpca", KernelPCA(
        kernel="rbf",
        eigen_solver="randomized",
        random_state=0,
        n_components=best_kpca_params["kpca__n_components"],
        gamma=best_kpca_params["kpca__gamma"],
    )),
    ("lda", LinearDiscriminantAnalysis(n_components=9)),
])

start = time.perf_counter()
X_train_red = best_reducer.fit_transform(X_train_hog_kpca, y_train_img_kpca)
reducer_fit_time = time.perf_counter() - start
X_test_red = best_reducer.transform(X_test_hog_kpca)

print("\nReduced shapes (KPCA+LDA):", X_train_red.shape, X_test_red.shape)
print(f"Reducer fit time: {reducer_fit_time:.2f} sec")


Reduced shapes (KPCA+LDA): (1500, 9) (500, 9)
Reducer fit time: 6.40 sec


In [9]:
# 5) Classification after reduction: kNN
knn_param_grid_red = {
    "n_neighbors": [1, 3, 5, 7, 9, 11],
    "metric": ["euclidean", "manhattan"],
    "weights": ["uniform", "distance"],
}

print("\nRunning GridSearchCV for kNN on KPCA+LDA features...")
start = time.perf_counter()
knn_grid_red = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_param_grid_red,
    scoring="accuracy",
    cv=3,
    n_jobs=-1
)
knn_grid_red.fit(X_train_red, y_train_img_kpca)
knn_grid_time = time.perf_counter() - start

best_knn_red = knn_grid_red.best_estimator_
y_pred_knn_red = best_knn_red.predict(X_test_red)

knn_acc_red = accuracy_score(y_test_img_kpca, y_pred_knn_red)
knn_f1_red  = f1_score(y_test_img_kpca, y_pred_knn_red, average="macro")

print("\n=== BEST kNN on HOG → KPCA → LDA ===")
print("Best kNN params:", knn_grid_red.best_params_)
print(f"kNN grid time: {knn_grid_time:.2f} sec")
print(f"Test accuracy: {knn_acc_red:.4f}")
print(f"Test macro F1: {knn_f1_red:.4f}")

results.append({
    "name": f"KNN_HOG_KPCA+LDA_best_{knn_grid_red.best_params_}",
    "train_time": hog_kpca_time + kpca_grid_time + reducer_fit_time + knn_grid_time,
    "test_accuracy": knn_acc_red,
    "test_macro_f1": knn_f1_red,
    "kpca_best_params": best_kpca_params,
})


Running GridSearchCV for kNN on KPCA+LDA features...

=== BEST kNN on HOG → KPCA → LDA ===
Best kNN params: {'metric': 'euclidean', 'n_neighbors': 11, 'weights': 'uniform'}
kNN grid time: 1.07 sec
Test accuracy: 0.4620
Test macro F1: 0.4609


In [10]:
# 6) Classification after reduction: NCC
ncc_param_grid_red = {
    "metric": ["euclidean", "manhattan"],
    "shrink_threshold": [None, 0.05, 0.1, 0.5, 1.0],
}

print("\nRunning GridSearchCV for NCC on KPCA+LDA features...")
start = time.perf_counter()
ncc_grid_red = GridSearchCV(
    estimator=NearestCentroid(),
    param_grid=ncc_param_grid_red,
    scoring="accuracy",
    cv=3,
    n_jobs=-1
)
ncc_grid_red.fit(X_train_red, y_train_img_kpca)
ncc_grid_time = time.perf_counter() - start

best_ncc_red = ncc_grid_red.best_estimator_
y_pred_ncc_red = best_ncc_red.predict(X_test_red)

ncc_acc_red = accuracy_score(y_test_img_kpca, y_pred_ncc_red)
ncc_f1_red  = f1_score(y_test_img_kpca, y_pred_ncc_red, average="macro")

print("\n=== BEST NCC on HOG → KPCA → LDA ===")
print("Best NCC params:", ncc_grid_red.best_params_)
print(f"NCC grid time: {ncc_grid_time:.2f} sec")
print(f"Test accuracy: {ncc_acc_red:.4f}")
print(f"Test macro F1: {ncc_f1_red:.4f}")

results.append({
    "name": f"NCC_HOG_KPCA+LDA_best_{ncc_grid_red.best_params_}",
    "train_time": hog_kpca_time + kpca_grid_time + reducer_fit_time + ncc_grid_time,
    "test_accuracy": ncc_acc_red,
    "test_macro_f1": ncc_f1_red,
    "kpca_best_params": best_kpca_params,
})


Running GridSearchCV for NCC on KPCA+LDA features...

=== BEST NCC on HOG → KPCA → LDA ===
Best NCC params: {'metric': 'euclidean', 'shrink_threshold': 0.5}
NCC grid time: 0.05 sec
Test accuracy: 0.5100
Test macro F1: 0.5158
